In [1]:
# %%

#------------------------------------------------ Begin_Librairie ----------------------------------------

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

import pandas as pd

from time import sleep

import datetime

from bs4 import BeautifulSoup

from pandas import ExcelWriter

import os

import re

    

In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'DE BAFIN' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.3")

now=datetime.datetime.now()

filename = '{} data {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}" ## to decomment for the production environment
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



# %%

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder,

         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert

         }

chromeOptions.add_argument("--disable-search-engine-choice-screen")

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()



    

Running DE BAFIN Web Scraping Tool v.1.3


In [3]:

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------



regdict = {

    'DE BAFIN 14' : "Credit institutions (BA)",

    'DE BAFIN 15' : "Financial services institutions (BA)",

    'DE BAFIN 16' : "Asset management companies",

    'DE BAFIN 18' : "Externally managed investment companies",

    'DE BAFIN 19' : "Branches (BA)",

    'DE BAFIN 20' : "EEA credit institutions (BA)",

    'DE BAFIN 21' : "EU management companies",

    'DE BAFIN 22' : "Branches of EU management companies",

    'DE BAFIN 23' : "Representative offices (BA)",

    'DE BAFIN 24' : "Information service delivery (BA)",

    'DE BAFIN 25' : "Exempted companies (BA)",

    'DE BAFIN 26' : "All insurers/ Pension funds",

    'DE BAFIN 27' : "Life insurers (VA)",

    'DE BAFIN 28' : "Health insurers",

    'DE BAFIN 29' : "Property and casualty insurers (VA)",

    'DE BAFIN 30' : "Pension funds (VA)",

    'DE BAFIN 31' : "Pensionskassen (VA)",

    'DE BAFIN 32' : "Reinsurers (VA)",

    'DE BAFIN 33' : "EEA service providers (VA)",

    'DE BAFIN 34' : "EEA branches (VA)",

    'DE BAFIN 35' : "Insurance undertakings and pension funds",

    'DE BAFIN 36' : "Insurers without business activities",

    'DE BAFIN 37' : "Credit services institutions (BA)",

    'DE BAFIN 38' : "Crypto Securities Registrars (BA)",

    'DE BAFIN 39' : "Crypto custodian (BA)",

    'DE BAFIN 40' : "Leasing/ Factoring institutions (BA)",

    'DE BAFIN 41' : "Securities institutions (BA)",   

}



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



ISO={"Afghanistan": "AF", "Ägypten": "EG", "Albanien": "AL", "Algerien": "DZ", 

     "Amerikanische Jungferninseln": "VI", "Amerikanisch-Ozeanien": "XA", 

     "Andorra": "AD", "Angola": "AO", "Anguilla": "AI", "Antigua und Barbuda": "AG", 

     "Äquatorialguinea": "GQ", "Arabische Republik Syrien": "SY", "Argentinien": "AR", 

     "Armenien": "AM", "Aruba": "AW", "Aserbaidschan": "AZ", "Äthiopien": "ET", 

     "Australien": "AU", "Australisch-Ozeanien": "XO", "Bahamas": "BS", "Bahrain": "BH", 

     "Bangladesch": "BD", "Barbados": "BB", "Belarus": "BY", "Belgien": "BE", "Belize": "BZ", 

     "Benin": "BJ", "Bermuda": "BM", "Bhutan": "BT", "Bolivien": "BO", "Bosnien und Herzegowina": "BA", 

     "Botsuana": "BW", "Brasilien": "BR", "Britische Jungferninseln": "VG", 

     "Britisches Gebiet im Indischen Ozean": "IO", "Brunei Darussalam": "BN", "Bulgarien": "BG", 

     "Burkina Faso": "BF", "Burundi": "BI", "Ceuta": "XC", "Chile": "CL", "China": "CN", "Costa Rica": "CR", 

     "Côte d'Ivoire": "CI", "Dänemark": "DK", "Demokratische Republik Kongo": "CD", 

     "Demokratische Volksrepublik Korea": "KP", "Demokratische Volksrepublik Laos": "LA", "Deutschland": "DE", 

     "Dominica": "DM", "Dominikanische Republik": "DO", "Dschibuti": "DJ", "Ecuador": "EC", 

     "Ehemalige Jugoslawische Republik Mazedonien": "MK", "El Salvador": "SV", "Eritrea": "ER", "Estland": "EE", 

     "Falklandinseln": "FK", "Färöer": "FO", "Fidschi": "FJ", "Finnland": "FI", 

     "Föderierte Staaten von Mikronesien": "FM", "Frankreich": "FR", "Französisch-Polynesien": "PF", "Gabun": "GA", 

     "Gambia": "GM", "Georgien": "GE", "Ghana": "GH", "Gibraltar": "GI", "Grenada": "GD", "Griechenland": "GR", 

     "Grönland": "GL", "Großbritannien": "GB", "Guam": "GU", "Guatemala": "GT", "Guinea": "GN", "Guinea-Bissau": "GW", 

     "Guyana": "GY", "Haiti": "HT", "Honduras": "HN", "Hongkong": "HK", "Indien": "IN", "Indonesien": "ID", 

     "Irak": "IQ", "Irland": "IE", "Islamische Republik Iran": "IR", "Island": "IS", "Israel": "IL", "Italien": "IT", 

     "Jamaika": "JM", "Japan": "JP", "Jemen": "YE", "Jordanien": "JO", "Jugoslawien": "YU", "Kaimaninseln": "KY", 

     "Kambodscha": "KH", "Kamerun": "CM", "Kanada": "CA", "Kap Verde": "CV", "Kasachstan": "KZ", "Katar": "QA", 

     "Kenia": "KE", "Kirgisische Republik": "KG", "Kiribati": "KI", "Kolumbien": "CO", "Komoren": "KM", "Kosovo": "XK", 

     "Kroatien": "HR", "Kuba": "CU", "Kuwait": "KW", "Lesotho": "LS", "Lettland": "LV", "Libanon": "LB", 

     "Liberia": "LR", "Libysch-Arabische Dschamahirija": "LY", "Liechtenstein": "LI", "Litauen": "LT", 

     "Luxemburg": "LU", "Macau": "MO", "Madagaskar": "MG", "Malawi": "MW", "Malaysia": "MY", "Malediven": "MV", 

     "Mali": "ML", "Malta": "MT", "Marokko": "MA", "Marshallinseln": "MH", "Mauretanien": "MR", "Mauritius": "MU", 

     "Mayotte": "YT", "Melilla": "XL", "Mexiko": "MX", "Mongolei": "MN", "Montenegro": "ME", "Montserrat": "MS", 

     "Mosambik": "MZ", "Myanmar": "MM", "Namibia": "NA", "Nauru": "NR", "Nepal": "NP", "Neukaledonien": "NC", 

     "Neuseeland": "NZ", "Neuseeländisch-Ozeanien": "XZ", "Nicaragua": "NI", "Nicht ermittelte Länder und Gebiete": "QU", 

     "Niederlande": "NL", "Niederländische Antillen": "AN", "Niger": "NE", "Nigeria": "NG", "Nördliche Marianen": "MP", 

     "Norwegen": "NO", "Oman": "OM", "Österreich": "AT", "Pakistan": "PK", "Palau": "PW", "Panama": "PA", 

     "Papua-Neuguinea": "PG", "Paraguay": "PY", "Peru": "PE", "Philippinen": "PH", "Pitcairn": "PN", 

     "Polargebiete": "XR", "Polen": "PL", "Portugal": "PT", "Republik Kongo": "CG", "Republik Korea": "KR", 

     "Republik Moldau": "MD", "Ruanda": "RW", "Rumänien": "RO", "Russische Föderation": "RU", "Salomonen": "SB", 

     "Sambia": "ZM", "Samoa": "WS", "San Marino": "SM", "São Tomé und Principe": "ST", "Saudi-Arabien": "SA", 

     "Schiffs- und Luftfahrzeugbedarf (Einfuhr auf deutsche und Ausfuhr bzw. Durchfuhr auf fremde Seeschiffe und Luftfahrzeuge)": "QR", 

     "Schweden": "SE", "Schweiz": "CH", "Senegal": "SN", "Serbien": "XS", "Seychellen": "SC", "Sierra Leone": "SL", 

     "Simbabwe": "ZW", "Singapur": "SG", "Slowakei": "SK", "Slowenien": "SI", "Somalia": "SO", "Spanien": "ES", 

     "Sri Lanka": "LK", "St. Helena": "SH", "St. Kitts und Nevis": "KN", "St. Lucia": "LC", "St. Pierre und Miquelon": "PM", 

     "St. Vincent": "VC", "Südafrika": "ZA", "Sudan": "SD", "Surinam": "SR", "Swasiland": "SZ", "Tadschikistan": "TJ", 

     "Taiwan": "TW", "Thailand": "TH", "Togo": "TG", "Tonga": "TO", "Trinidad und Tobago": "TT", "Tschad": "TD", 

     "Tschechische Republik": "CZ", "Tunesien": "TN", "Türkei": "TR", "Turkmenistan": "TM", "Turks- und Caicosinseln": "TC",

     "Tuvalu": "TV", "Uganda": "UG", "Ukraine": "UA", "Ungarn": "HU", "Uruguay": "UY", "Usbekistan": "UZ", "Vanuatu": "VU", 

     "Vatikanstadt": "VA", "Venezuela": "VE", "Vereinigte Arabische Emirate": "AE", "Vereinigte Republik Tansania": "TZ", 

     "Vereinigte Staaten": "US", "Vietnam": "VN", "Wallis und Futuna": "WF", "Westjordanland/Gazastreifen": "XP", 

     "Zentralafrikanische Republik": "CF", "Zypern": "CY",'Vereinigte Staaten von Amerika': 'US', 

     'Brit. Jungferninseln': 'VG', 'Cayman Inseln (Kaimaninseln)': 'KY', 'Jersey, Kanalinseln': 'JE',

     'Republik Singapur': 'SG'}



pattern = re.compile('([0-9]+)')

processdate = now.strftime('%Y-%m-%d')





In [ ]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    len_value=[]

    for key, value in sqldict.items():

        len_value.append(len(value))

    maxlen = max(len_value)

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



def check_dowload_files(tempfolder, fileType, wait_time=10):

    for time in range(wait_time):

        if len([ele for ele in os.listdir(tempfolder) if '.crdownload' not in ele and '.tmp' not in ele]) != 0 :

            print(f"[INFO] : - {fileType} file = {os.listdir(tempfolder)})")

            break

        else:

            print(f"[INFO] : -- Download {fileType} file ... (wait {time*2}/20 s)")

            sleep(2)

    else:

        raise Exception(f'[ERROR] : - Failed to Download {fileType} file. Run Script again' )

    return  os.listdir(tempfolder)[0]



def scrollinAndClick(xpath,key_press=False):

    if len(xpath) != 0 :

        for times in range(60):

            try:

                driver.find_element(By.XPATH, xpath).click()

                sleep(1)

                break

            except:

                print(f"[ERROR] : trying {times+1}/60 to key press 'DOWN' (scrolling)")

                sleep(1)

                if key_press:                    

                    driver.find_element(By.TAG_NAME, 'body').send_keys(key_press)

                    #driver.find_element(By.TAG_NAME, 'body').send_keys(Keys.DOWN)

        else:   

            raise Exception(f'[ERROR] : Failed scrollin Or Click on xpath element : {xpath}')





In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

driver.get('https://portal.mvp.bafin.de/database/InstInfo/?locale=en_GB')

sleep(3)



for k, reg in enumerate(regdict):

	print(f"[INFO] : {k+1}/{len(regdict)} [{reg}]  | {regdict[reg]}")

	driver.find_element(By.XPATH, f'//*[@id="institutKategorie"]/option[contains(text(),"{regdict[reg]}")]').click() # selecte cathegorie

	sleep(2)

	driver.find_element(By.XPATH, '//*[@id="sucheButtonInstitut"]').click() # click schet Button Institut

	try:
		scrollinAndClick('//*[@id="content"]//*/span[contains(text(),"CSV")]',Keys.DOWN)  # select CSV file
	except:
		print(f"[ERROR] : No CSV file to download for {reg} | {regdict[reg]}")
		continue
	sleep(1)

	file =  check_dowload_files(tempfolder, "csv" )

	filePath = os.path.join(tempfolder, file)

	df = pd.read_csv(filePath, sep=";", low_memory=False)  # read csv file

	df = df[df['NAME'].notna() & (df['NAME'] != '')]

	df = df.reset_index(drop=True)

	df = df.fillna("")  # subsitute nan with empty strings

	print(f"[INFO] : - DataFrame '{file}' | containe = {df.shape}")



	for index, row in df.iterrows():

		sqldict['Name'].append(row['NAME'].strip())

		sqldict['LEI Code'].append(row['LEI'].strip())

		sqldict['InternalID_1'].append(row['BAK NR']) if str(row['BAK NR']).find('-') == -1 else sqldict['InternalID_1'].append('')

		sqldict['InternalID_1_type'].append('BAK NR') if str(row['BAK NR']).find('-') == -1 else sqldict['InternalID_1_type'].append('')



		sqldict['InternalID_2'].append(row['BAFIN-ID'])

		sqldict['InternalID_2_type'].append('BAFIN-ID')



		sqldict['RegulationType'].append('Regulated')

		sqldict['Typology'].append(regdict[reg])

		sqldict['ListProcessDate'].append(processdate)

		sqldict['RegCtry'].append(reg.split(' ')[0])

		sqldict['RegCode'].append(reg.split(' ')[1])

		sqldict['ListCode'].append(reg.split(' ')[-1])

		sqldict['Website'].append(row['DISPUTE RESOLUTION ENTITY'].strip())



		sqldict['Address_1'].append(str(row['STREET']).strip())

		sqldict['Zip'].append(str(row['ZIP']).strip())

		sqldict['City'].append(str(row['CITY']).strip())

		sqldict['Cntry'].append(ISO.get(row['COUNTRY'].strip()))

		

		sqldict = bourange_same_length_array(sqldict)

			

	for rem in os.listdir(tempfolder):

		os.remove(os.path.join(tempfolder, rem))

	sleep(1)




[INFO] : 1/27 [DE BAFIN 14]  | Credit institutions (BA)
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (1236, 20)
[INFO] : 2/27 [DE BAFIN 15]  | Financial services institutions (BA)
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (1035, 20)
[INFO] : 3/27 [DE BAFIN 16]  | Asset management companies
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (698, 20)
[INFO] : 4/27 [DE BAFIN 18]  | Externally managed investment companies
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (3, 20)
[INFO] : 5/27 [DE BAFIN 19]  | Branches (BA)
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (161, 20)
[INFO] : 6/27 [DE BAFIN 20]  | EEA credit institutions (BA)
[INFO] : - csv file = ['export.csv'])
[INFO] : - DataFrame 'export.csv' | containe = (889, 20)
[INFO] : 7/27 [DE BAFIN 21]  | EU management companies
[INFO] : - csv fi

In [6]:

# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df.to_excel(filename, 'SQL Ready', index=False)


driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\1\ipykernel_48404\3116231628.py:9: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)
